In [1]:
import os
print(os.getcwd())

C:\Users\1111\Documents\LAB2


In [2]:
%%writefile app_final.py
import sqlite3
import os
import uuid
from flask import Flask, render_template, request, url_for, flash, redirect
from werkzeug.exceptions import abort
from werkzeug.utils import secure_filename
from PIL import Image

UPLOAD_FOLDER = 'static/uploads'
THUMBNAIL_FOLDER = 'static/thumbnails'
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif'}
MAX_FILE_SIZE = 5 * 1024 * 1024
THUMBNAIL_SIZE = (300, 200)

app = Flask(__name__)
app.config['SECRET_KEY'] = 'your secret key'
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['THUMBNAIL_FOLDER'] = THUMBNAIL_FOLDER
app.config['MAX_CONTENT_LENGTH'] = MAX_FILE_SIZE

os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
os.makedirs(app.config['THUMBNAIL_FOLDER'], exist_ok=True)

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def create_thumbnail(filepath, thumbpath):
    try:
        img = Image.open(filepath)
        img.thumbnail(THUMBNAIL_SIZE)
        img.save(thumbpath)
        return True
    except Exception as e:
        print(f"Ошибка: {e}")
        return False

def get_db_connection():
    conn = sqlite3.connect('database.db')
    conn.row_factory = sqlite3.Row
    return conn

def get_post(post_id):
    conn = get_db_connection()
    post = conn.execute('SELECT * FROM posts WHERE id = ?', (post_id,)).fetchone()
    conn.close()
    if post is None:
        abort(404)
    return post

@app.route('/')
def index():
    page = request.args.get('page', 1, type=int)
    per_page = 5
    offset = (page - 1) * per_page
    conn = get_db_connection()
    total = conn.execute('SELECT COUNT(*) FROM posts').fetchone()[0]
    posts = conn.execute('SELECT * FROM posts ORDER BY created DESC LIMIT ? OFFSET ?', (per_page, offset)).fetchall()
    conn.close()
    total_pages = (total + per_page - 1) // per_page
    return render_template('index.html', posts=posts, page=page, total_pages=total_pages)

@app.route('/<int:post_id>')
def post(post_id):
    post = get_post(post_id)
    return render_template('post.html', post=post)

@app.route('/create', methods=('GET', 'POST'))
def create():
    if request.method == 'POST':
        title = request.form['title']
        content = request.form['content']
        file = request.files.get('image')
        if not title:
            flash('Title is required!')
        else:
            image_filename = None
            if file and allowed_file(file.filename):
                original_name = secure_filename(file.filename)
                unique_name = f"{uuid.uuid4().hex}_{original_name}"
                filepath = os.path.join(app.config['UPLOAD_FOLDER'], unique_name)
                file.save(filepath)
                thumb_filename = f"thumb_{unique_name}"
                thumbpath = os.path.join(app.config['THUMBNAIL_FOLDER'], thumb_filename)
                if create_thumbnail(filepath, thumbpath):
                    image_filename = unique_name
                else:
                    image_filename = unique_name
            conn = get_db_connection()
            conn.execute('INSERT INTO posts (title, content, image) VALUES (?, ?, ?)', (title, content, image_filename))
            conn.commit()
            conn.close()
            return redirect(url_for('index'))
    return render_template('create.html')

@app.route('/<int:id>/edit', methods=('GET', 'POST'))
def edit(id):
    post = get_post(id)
    if request.method == 'POST':
        title = request.form['title']
        content = request.form['content']
        file = request.files.get('image')
        if not title:
            flash('Title is required!')
        else:
            image_filename = post['image']
            if file and allowed_file(file.filename):
                if post['image']:
                    old_path = os.path.join(app.config['UPLOAD_FOLDER'], post['image'])
                    old_thumb = os.path.join(app.config['THUMBNAIL_FOLDER'], f"thumb_{post['image']}")
                    if os.path.exists(old_path): os.remove(old_path)
                    if os.path.exists(old_thumb): os.remove(old_thumb)
                original_name = secure_filename(file.filename)
                unique_name = f"{uuid.uuid4().hex}_{original_name}"
                filepath = os.path.join(app.config['UPLOAD_FOLDER'], unique_name)
                file.save(filepath)
                thumb_filename = f"thumb_{unique_name}"
                thumbpath = os.path.join(app.config['THUMBNAIL_FOLDER'], thumb_filename)
                if create_thumbnail(filepath, thumbpath):
                    image_filename = unique_name
                else:
                    image_filename = unique_name
            conn = get_db_connection()
            conn.execute('UPDATE posts SET title = ?, content = ?, image = ? WHERE id = ?', (title, content, image_filename, id))
            conn.commit()
            conn.close()
            return redirect(url_for('index'))
    return render_template('edit.html', post=post)

@app.route('/<int:id>/delete', methods=('POST',))
def delete(id):
    post = get_post(id)
    if post['image']:
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], post['image'])
        thumbpath = os.path.join(app.config['THUMBNAIL_FOLDER'], f"thumb_{post['image']}")
        if os.path.exists(filepath): os.remove(filepath)
        if os.path.exists(thumbpath): os.remove(thumbpath)
    conn = get_db_connection()
    conn.execute('DELETE FROM posts WHERE id = ?', (id,))
    conn.commit()
    conn.close()
    flash(f'"{post["title"]}" was successfully deleted!')
    return redirect(url_for('index'))

@app.route('/search')
def search():
    query = request.args.get('q', '').strip()
    if not query:
        return redirect(url_for('index'))
    page = request.args.get('page', 1, type=int)
    per_page = 5
    offset = (page - 1) * per_page
    conn = get_db_connection()
    total = conn.execute("SELECT COUNT(*) FROM posts WHERE title LIKE ? OR content LIKE ?", (f'%{query}%', f'%{query}%')).fetchone()[0]
    posts = conn.execute("SELECT * FROM posts WHERE title LIKE ? OR content LIKE ? ORDER BY created DESC LIMIT ? OFFSET ?",
                         (f'%{query}%', f'%{query}%', per_page, offset)).fetchall()
    conn.close()
    total_pages = (total + per_page - 1) // per_page
    return render_template('search_results.html', posts=posts, query=query, page=page, total_pages=total_pages)

if __name__ == '__main__':
    app.run(debug=True)

Overwriting app_final.py


In [3]:
%%writefile schema.sql
DROP TABLE IF EXISTS posts;

CREATE TABLE posts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    created TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    title TEXT NOT NULL,
    content TEXT NOT NULL,
    image TEXT
);

Overwriting schema.sql


In [4]:
import os
os.makedirs('static/uploads', exist_ok=True)
os.makedirs('static/thumbnails', exist_ok=True)
print("Папки созданы")

Папки созданы


In [5]:
!pip install pillow

In [6]:
!python init_db.py

База данных успешно создана и заполнена тестовыми данными!


In [7]:
%%writefile init_db.py
import sqlite3

connection = sqlite3.connect('database.db')
with open('schema.sql') as f:
    connection.executescript(f.read())
cur = connection.cursor()
cur.execute("INSERT INTO posts (title, content) VALUES (?, ?)",
            ('First Post', 'Content for the first post'))
cur.execute("INSERT INTO posts (title, content) VALUES (?, ?)",
            ('Second Post', 'Content for the second post'))
connection.commit()
connection.close()
print("База данных успешно создана и заполнена тестовыми данными!")

Overwriting init_db.py


In [8]:
!python init_db.py

База данных успешно создана и заполнена тестовыми данными!


In [9]:
%%writefile templates/base.html
<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.3/dist/css/bootstrap.min.css" rel="stylesheet">
    <title>{% block title %} {% endblock %}</title>
  </head>
  <body>
    <nav class="navbar navbar-expand-lg bg-light">
      <div class="container-fluid">
        <a class="navbar-brand" href="{{ url_for('index') }}">FlaskBlog</a>
        <button class="navbar-toggler" type="button" data-bs-toggle="collapse" data-bs-target="#navbarSupportedContent">
          <span class="navbar-toggler-icon"></span>
        </button>
        <div class="collapse navbar-collapse" id="navbarSupportedContent">
          <ul class="navbar-nav me-auto mb-2 mb-lg-0">
            <li class="nav-item">
              <a class="nav-link" href="{{ url_for('create') }}">New Post</a>
            </li>
          </ul>
          <form class="d-flex" action="{{ url_for('search') }}" method="get">
            <input class="form-control me-2" type="search" name="q" placeholder="Search posts" aria-label="Search">
            <button class="btn btn-outline-success" type="submit">Search</button>
          </form>
        </div>
      </div>
    </nav>
    <div class="container">
      {% for message in get_flashed_messages() %}
        <div class="alert alert-danger">{{ message }}</div>
      {% endfor %}
      {% block content %} {% endblock %}
    </div>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.3/dist/js/bootstrap.bundle.min.js"></script>
  </body>
</html>

Overwriting templates/base.html


In [10]:
%%writefile templates/index.html
{% extends 'base.html' %}

{% block content %}
    <h1>{% block title %} Welcome to FlaskBlog {% endblock %}</h1>
    {% for post in posts %}
        <div class="card mb-3">
            <div class="row g-0">
                {% if post['image'] %}
                <div class="col-md-4">
                    <img src="{{ url_for('static', filename='thumbnails/thumb_' + post['image']) }}"
                         class="img-fluid rounded-start" alt="Thumbnail">
                </div>
                <div class="col-md-8">
                {% else %}
                <div class="col-md-12">
                {% endif %}
                    <div class="card-body">
                        <h5 class="card-title">
                            <a href="{{ url_for('post', post_id=post['id']) }}">{{ post['title'] }}</a>
                        </h5>
                        <p class="card-text">{{ post['content'][:200] }}{% if post['content']|length > 200 %}...{% endif %}</p>
                        <p class="card-text"><small class="text-muted">{{ post['created'] }}</small></p>
                        <a href="{{ url_for('edit', id=post['id']) }}" class="btn btn-warning btn-sm">Edit</a>
                    </div>
                </div>
            </div>
        </div>
    {% endfor %}

    <!-- Пагинация -->
    <nav aria-label="Page navigation">
        <ul class="pagination justify-content-center">
            {% if page > 1 %}
                <li class="page-item"><a class="page-link" href="{{ url_for('index', page=page-1) }}">Previous</a></li>
            {% else %}
                <li class="page-item disabled"><span class="page-link">Previous</span></li>
            {% endif %}

            {% for p in range(1, total_pages+1) %}
                {% if p == page %}
                    <li class="page-item active"><span class="page-link">{{ p }}</span></li>
                {% else %}
                    <li class="page-item"><a class="page-link" href="{{ url_for('index', page=p) }}">{{ p }}</a></li>
                {% endif %}
            {% endfor %}

            {% if page < total_pages %}
                <li class="page-item"><a class="page-link" href="{{ url_for('index', page=page+1) }}">Next</a></li>
            {% else %}
                <li class="page-item disabled"><span class="page-link">Next</span></li>
            {% endif %}
        </ul>
    </nav>
{% endblock %}

Overwriting templates/index.html


In [11]:
%%writefile templates/post.html
{% extends 'base.html' %}

{% block content %}
    <h2>{% block title %} {{ post['title'] }} {% endblock %}</h2>
    {% if post['image'] %}
        <img src="{{ url_for('static', filename='uploads/' + post['image']) }}" class="img-fluid mb-3" alt="Post image">
    {% endif %}
    <p>{{ post['content'] }}</p>
    <span class="badge text-bg-primary">{{ post['created'] }}</span>
    <a href="{{ url_for('edit', id=post['id']) }}" class="btn btn-warning btn-sm">Edit</a>
{% endblock %}

Overwriting templates/post.html


In [12]:
%%writefile templates/create.html
{% extends 'base.html' %}

{% block content %}
<h1>{% block title %} Create a New Post {% endblock %}</h1>

<form method="post" enctype="multipart/form-data">
  <div class="mb-3">
    <label for="title" class="form-label">Title</label>
    <input type="text" name="title" placeholder="Post title" class="form-control" value="{{ request.form['title'] }}">
  </div>

  <div class="mb-3">
    <label for="content" class="form-label">Content</label>
    <textarea name="content" placeholder="Post content" class="form-control" rows="3">{{ request.form['content'] }}</textarea>
  </div>

  <div class="mb-3">
    <label for="image" class="form-label">Image (optional)</label>
    <input type="file" name="image" accept="image/*" class="form-control">
    <div class="form-text">Allowed formats: PNG, JPG, JPEG, GIF. Max size 5 MB.</div>
  </div>

  <button type="submit" class="btn btn-primary">Submit</button>
</form>
{% endblock %}

Overwriting templates/create.html


In [13]:
%%writefile templates/edit.html
{% extends 'base.html' %}

{% block content %}
<h1>{% block title %} Edit "{{ post['title'] }}" {% endblock %}</h1>

<form method="post" enctype="multipart/form-data">
  <div class="mb-3">
    <label for="title" class="form-label">Title</label>
    <input type="text" name="title" placeholder="Post title" class="form-control" value="{{ request.form['title'] or post['title'] }}">
  </div>

  <div class="mb-3">
    <label for="content" class="form-label">Content</label>
    <textarea name="content" placeholder="Post content" class="form-control" rows="3">{{ request.form['content'] or post['content'] }}</textarea>
  </div>

  <div class="mb-3">
    <label for="image" class="form-label">Image (optional)</label>
    {% if post['image'] %}
        <div class="mb-2">
            <img src="{{ url_for('static', filename='uploads/' + post['image']) }}" style="max-height: 100px;" alt="Current image">
            <p class="small">Current image. Upload a new one to replace it.</p>
        </div>
    {% endif %}
    <input type="file" name="image" accept="image/*" class="form-control">
    <div class="form-text">Allowed formats: PNG, JPG, JPEG, GIF. Max size 5 MB.</div>
  </div>

  <button type="submit" class="btn btn-primary">Submit</button>
</form>
<hr>
<form action="{{ url_for('delete', id=post['id']) }}" method="POST">
  <input type="submit" value="Delete Post" class="btn btn-danger btn-sm" onclick="return confirm('Are you sure you want to delete this post?')">
</form>
{% endblock %}

Overwriting templates/edit.html


In [14]:
%%writefile templates/search_results.html
{% extends 'base.html' %}

{% block content %}
    <h1>Search results for "{{ query }}"</h1>
    {% if posts %}
        {% for post in posts %}
        <div class="card mb-3">
            <div class="row g-0">
                {% if post['image'] %}
                <div class="col-md-4">
                    <img src="{{ url_for('static', filename='thumbnails/thumb_' + post['image']) }}"
                         class="img-fluid rounded-start" alt="Thumbnail">
                </div>
                <div class="col-md-8">
                {% else %}
                <div class="col-md-12">
                {% endif %}
                    <div class="card-body">
                        <h5 class="card-title">
                            <a href="{{ url_for('post', post_id=post['id']) }}">{{ post['title'] }}</a>
                        </h5>
                        <p class="card-text">{{ post['content'][:200] }}{% if post['content']|length > 200 %}...{% endif %}</p>
                        <p class="card-text"><small class="text-muted">{{ post['created'] }}</small></p>
                        <a href="{{ url_for('edit', id=post['id']) }}" class="btn btn-warning btn-sm">Edit</a>
                    </div>
                </div>
            </div>
        </div>
        {% endfor %}

        <nav aria-label="Page navigation">
            <ul class="pagination justify-content-center">
                {% if page > 1 %}
                    <li class="page-item"><a class="page-link" href="{{ url_for('search', q=query, page=page-1) }}">Previous</a></li>
                {% else %}
                    <li class="page-item disabled"><span class="page-link">Previous</span></li>
                {% endif %}

                {% for p in range(1, total_pages+1) %}
                    {% if p == page %}
                        <li class="page-item active"><span class="page-link">{{ p }}</span></li>
                    {% else %}
                        <li class="page-item"><a class="page-link" href="{{ url_for('search', q=query, page=p) }}">{{ p }}</a></li>
                    {% endif %}
                {% endfor %}

                {% if page < total_pages %}
                    <li class="page-item"><a class="page-link" href="{{ url_for('search', q=query, page=page+1) }}">Next</a></li>
                {% else %}
                    <li class="page-item disabled"><span class="page-link">Next</span></li>
                {% endif %}
            </ul>
        </nav>
    {% else %}
        <p>No posts found matching your query.</p>
    {% endif %}
{% endblock %}

Writing templates/search_results.html
